[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/071.%20The%20Classic%20Vehicle%20Routing%20Problem%20%28VRP%29/P71-Tier-9_executed.ipynb)

---

# P71-Tier-9: A QAOA State-Vector Study of a Vehicle-Assignment Model

## Problem, purpose, and scope

This notebook uses a small vehicle-routing instance to teach the main steps of a **QAOA-style variational optimization workflow**: construct a discrete cost model, prepare a superposition, alternate cost and mixing operations, optimize their parameters with a classical optimizer, and interpret samples from the final state.

The model assigns each customer to one of a fixed number of vehicles. Customers assigned to the same vehicle are visited in customer-index order. The cost includes the resulting route distance and penalties for invalid assignments or vehicle-capacity violations. This is a deliberately restricted routing model: it does not optimize the order of visits, use time windows, or model vehicle-specific costs.

The assignment and capacity cost is evaluated as a diagonal cost operator over a compact binary encoding. It is **not a QUBO formulation**: exact route-distance and capacity costs in this encoding are not generally quadratic in the bits. QAOA does not require every cost operator to be a QUBO, but the distinction matters when interpreting the model. The simulation is classical state-vector computation; it does not use quantum hardware and does not establish quantum advantage.

### Inputs and outputs

The inputs are the depot and customer coordinates, customer demands, vehicle capacity, and maximum number of vehicles. The notebook reports sampled feasible assignments, their route distances, the optimized circuit parameters, and a comparison with a simple feasible greedy construction. Results depend on the optimizer and finite sampling; they are not certificates of global optimality.

### Background: QAOA in this notebook

A qubit has computational-basis states $\lvert 0\rangle$ and $\lvert 1\rangle$. A state-vector simulator stores a complex amplitude for each basis state. If the state is

$$
\lvert\psi\rangle = \sum_z a_z\lvert z\rangle,
\qquad \sum_z |a_z|^2 = 1,
$$

then measuring it returns bit string $z$ with probability $|a_z|^2$. A **cost unitary** changes each amplitude's phase according to its cost, while a **mixer unitary** moves amplitude between bit strings. A classical optimizer adjusts the angles of these operations to reduce the expected cost.

For $p$ layers, the state has the form

$$
\lvert\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
=
\left[\prod_{\ell=1}^{p}
U_M(\beta_\ell)U_C(\gamma_\ell)\right]
\lvert +\rangle^{\otimes n},
\qquad
U_C(\gamma)=e^{-i\gamma C},
\qquad
U_M(\beta)=e^{-i\beta\sum_{q=0}^{n-1}X_q}.
$$

Here $C$ is the diagonal cost operator, $X_q$ flips qubit $q$, and $\lvert +\rangle=(\lvert0\rangle+\lvert1\rangle)/\sqrt{2}$. The classical objective is the expected cost $\langle\psi|C|\psi\rangle$. This notebook simulates these operations directly and uses `scipy.optimize.minimize` to select the angles. With $n$ qubits, the state vector has $2^n$ amplitudes; exact state-vector memory and gate work therefore grow exponentially with $n$.

In [ ]:
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

np.random.seed(71)
plt.style.use("seaborn-v0_8")


## 1. Define the assignment and cost model

Each customer is represented by two bits. The codes `00`, `01`, and `10` select vehicles 0, 1, and 2; `11` is invalid and receives a penalty. This compact encoding uses two qubits per customer rather than one qubit for every customer–vehicle pair.

For an assignment, customers assigned to each vehicle are visited in their listed order. The cost is the sum of these route distances plus a penalty for each vehicle whose total demand exceeds capacity. The penalty makes infeasible assignments expensive, but it does not mathematically guarantee that the minimum-cost state is feasible. Feasibility is checked explicitly when samples are decoded.

In [ ]:
class QuantumState:
    """State vector and computational-basis measurements for a small simulator."""

    def __init__(self, amplitudes: np.ndarray, basis_states: List[str]):
        self.amplitudes = np.asarray(amplitudes, dtype=complex)
        self.basis_states = basis_states
        self.dimension = len(basis_states[0])
        probabilities = self.probabilities()
        if not np.isclose(probabilities.sum(), 1.0, atol=1e-10):
            raise ValueError("State amplitudes must have unit total probability.")

    def probabilities(self) -> np.ndarray:
        """Return the measurement probability of each computational basis state."""
        return np.abs(self.amplitudes) ** 2

    def measure(self) -> str:
        """Draw one bit string according to the state's Born probabilities."""
        index = np.random.choice(len(self.basis_states), p=self.probabilities())
        return self.basis_states[index]

    def expectation_value(self, diagonal_costs: np.ndarray) -> float:
        """Evaluate the expectation of a diagonal cost operator."""
        return float(np.dot(self.probabilities(), diagonal_costs))


class QuantumInspiredQAOA:
    """Classical state-vector simulation of a QAOA assignment model."""

    def __init__(
        self,
        depot_location: Tuple[float, float],
        customer_locations: List[Tuple[float, float]],
        customer_demands: List[int],
        vehicle_capacity: int,
        max_vehicles: int = 3,
    ):
        if len(customer_locations) != len(customer_demands):
            raise ValueError("Each customer must have exactly one demand value.")
        if max_vehicles != 3:
            raise ValueError("The two-bit vehicle code in this example supports three vehicles.")
        if vehicle_capacity <= 0 or any(demand < 0 for demand in customer_demands):
            raise ValueError("Capacity must be positive and demands must be non-negative.")

        self.depot_location = depot_location
        self.customer_locations = customer_locations
        self.customer_demands = customer_demands
        self.vehicle_capacity = vehicle_capacity
        self.max_vehicles = max_vehicles
        self.n_customers = len(customer_locations)
        self.n_qubits = 2 * self.n_customers
        self.n_layers = 2
        self.distances = self._calculate_distance_matrix()
        self.basis_states = [format(i, f"0{self.n_qubits}b") for i in range(2**self.n_qubits)]
        self.costs = self._build_cost_vector()

    def _calculate_distance_matrix(self) -> np.ndarray:
        """Calculate Euclidean distances between the depot and customers."""
        locations = [self.depot_location] + self.customer_locations
        return np.linalg.norm(
            np.asarray(locations)[:, None, :] - np.asarray(locations)[None, :, :],
            axis=2,
        )

    def decode_assignment(self, bit_string: str) -> List[int]:
        """Decode each customer's two-bit code; code 3 denotes an invalid choice."""
        if len(bit_string) != self.n_qubits:
            raise ValueError("Bit-string length does not match the number of qubits.")
        return [
            int(bit_string[2 * i : 2 * i + 2], 2)
            for i in range(self.n_customers)
        ]

    def _assignment_cost(self, assignment: List[int]) -> float:
        """Return route distance plus explicit invalid-code and capacity penalties."""
        if any(vehicle >= self.max_vehicles for vehicle in assignment):
            return 1000.0 + sum(self.distances[0, i + 1] for i in range(self.n_customers))

        total_distance = 0.0
        total_penalty = 0.0
        for vehicle in range(self.max_vehicles):
            customers = [i for i, assigned_vehicle in enumerate(assignment) if assigned_vehicle == vehicle]
            if not customers:
                continue

            demand = sum(self.customer_demands[i] for i in customers)
            total_penalty += 1000.0 * max(0, demand - self.vehicle_capacity)
            route = [0] + [i + 1 for i in customers] + [0]
            total_distance += sum(
                self.distances[route[position], route[position + 1]]
                for position in range(len(route) - 1)
            )
        return total_distance + total_penalty

    def _build_cost_vector(self) -> np.ndarray:
        """Precompute the diagonal cost for every encoded assignment state."""
        return np.asarray(
            [self._assignment_cost(self.decode_assignment(state)) for state in self.basis_states],
            dtype=float,
        )

    def create_initial_quantum_state(self) -> QuantumState:
        """Prepare the uniform superposition over all binary encodings."""
        amplitudes = np.full(2**self.n_qubits, 1 / np.sqrt(2**self.n_qubits), dtype=complex)
        return QuantumState(amplitudes, self.basis_states.copy())

    @staticmethod
    def _apply_mixer(state: QuantumState, beta: float) -> QuantumState:
        """Apply exp(-i beta sum_q X_q) using independent one-qubit rotations."""
        amplitudes = state.amplitudes.copy()
        cosine = np.cos(beta)
        sine = -1j * np.sin(beta)
        indices = np.arange(len(amplitudes))

        for qubit in range(state.dimension):
            bit_mask = 1 << (state.dimension - 1 - qubit)
            zero_indices = indices[(indices & bit_mask) == 0]
            one_indices = zero_indices | bit_mask
            zero_values = amplitudes[zero_indices].copy()
            one_values = amplitudes[one_indices].copy()
            amplitudes[zero_indices] = cosine * zero_values + sine * one_values
            amplitudes[one_indices] = sine * zero_values + cosine * one_values

        return QuantumState(amplitudes, state.basis_states.copy())

    def qaoa_circuit(self, params: np.ndarray) -> QuantumState:
        """Apply alternating diagonal cost and transverse-field mixer layers."""
        if len(params) != 2 * self.n_layers:
            raise ValueError("There must be one cost angle and one mixer angle per layer.")
        state = self.create_initial_quantum_state()
        for layer in range(self.n_layers):
            gamma = params[2 * layer]
            beta = params[2 * layer + 1]
            state.amplitudes *= np.exp(-1j * gamma * self.costs)
            state = self._apply_mixer(state, beta)
        return state

    def objective_function(self, params: np.ndarray) -> float:
        """Return the expected assignment cost used by the classical optimizer."""
        return self.qaoa_circuit(params).expectation_value(self.costs)

    def optimize_parameters(
        self, max_iterations: int = 30
    ) -> Tuple[np.ndarray, float]:
        """Optimize QAOA angles with the classical L-BFGS-B optimizer."""
        initial_params = np.random.uniform(0, 2 * np.pi, 2 * self.n_layers)
        result = minimize(
            fun=self.objective_function,
            x0=initial_params,
            method="L-BFGS-B",
            bounds=[(0, 2 * np.pi)] * len(initial_params),
            options={"maxiter": max_iterations},
        )
        return result.x, float(result.fun)

    def calculate_route_distance(self, route: List[int]) -> float:
        """Calculate a depot-to-depot route distance for 1-based customer IDs."""
        if not route:
            return 0.0
        locations = [0] + route + [0]
        return float(sum(
            self.distances[locations[position], locations[position + 1]]
            for position in range(len(locations) - 1)
        ))

    def decode_quantum_solution(self, bit_string: str) -> Tuple[List[List[int]], bool]:
        """Convert a sampled bit string to routes and report capacity feasibility."""
        assignment = self.decode_assignment(bit_string)
        if any(vehicle >= self.max_vehicles for vehicle in assignment):
            return [], False

        routes = []
        feasible = True
        for vehicle in range(self.max_vehicles):
            route = [i + 1 for i, assigned_vehicle in enumerate(assignment) if assigned_vehicle == vehicle]
            if route:
                demand = sum(self.customer_demands[customer - 1] for customer in route)
                feasible &= demand <= self.vehicle_capacity
                routes.append(route)
        return routes, feasible

    def solve_vrp_quantum(self, max_iterations: int = 30, n_measurements: int = 100) -> Dict:
        """Optimize circuit angles, sample the resulting state, and retain feasible samples."""
        start_time = __import__("time").time()
        optimal_params, expected_cost = self.optimize_parameters(max_iterations)
        optimization_time = __import__("time").time() - start_time
        final_state = self.qaoa_circuit(optimal_params)

        measurements: Dict[str, int] = {}
        feasible_samples = []
        for _ in range(n_measurements):
            measured = final_state.measure()
            measurements[measured] = measurements.get(measured, 0) + 1
            routes, feasible = self.decode_quantum_solution(measured)
            if feasible:
                distance = sum(self.calculate_route_distance(route) for route in routes)
                feasible_samples.append((distance, measured, routes))

        if feasible_samples:
            total_distance, best_state, routes = min(feasible_samples, key=lambda item: item[0])
            best_frequency = measurements[best_state] / n_measurements
        else:
            total_distance, best_state, routes, best_frequency = float("nan"), None, [], 0.0

        return {
            "routes": routes,
            "total_distance": total_distance,
            "total_cost": total_distance * 2.5,
            "optimal_params": optimal_params,
            "expected_cost": expected_cost,
            "optimization_time": optimization_time,
            "best_state": best_state,
            "best_frequency": best_frequency,
            "measurements": measurements,
            "final_state": final_state,
            "feasible_sample_count": len(feasible_samples),
        }


### Mathematical model and code connection

Let $i\in\{1,\ldots,n\}$ index customers and $k\in\{0,1,2\}$ index vehicles. A valid assignment chooses exactly one vehicle for every customer. The two-bit code stores this choice directly; a code of `11` is invalid. For vehicle $k$, let $R_k$ be the customers assigned to it, in increasing customer-index order, and let $d_{ij}$ be the Euclidean distance between locations $i$ and $j$, with location 0 denoting the depot.

The modeled route distance and penalized assignment cost are

$$
D(R_k)=
\begin{cases}
0, & R_k=\varnothing,\\
d_{0,r_1}+\displaystyle\sum_{j=1}^{m-1}d_{r_j,r_{j+1}}+d_{r_m,0},
&R_k=(r_1,\ldots,r_m),
\end{cases}
$$

$$
C(a)=\sum_{k=0}^{2}D(R_k)
+1000\sum_{k=0}^{2}\max\!\left(0,\sum_{i\in R_k}q_i-Q\right)
+P_{\mathrm{invalid}}(a).
$$

Here $a_i$ is customer $i$'s decoded vehicle, $q_i$ is its demand, $Q$ is vehicle capacity, and $P_{\mathrm{invalid}}$ is the penalty returned for any invalid two-bit code. The numerical penalty scale is part of this example's model; feasibility is still checked independently. In the code, `_assignment_cost` evaluates $C(a)$ and `_build_cost_vector` stores $C$ for every binary basis state. Thus the cost operator is diagonal, with its diagonal entries in `self.costs`.

The circuit starts with equal probability on all $2^n$ bit strings. Each cost operation multiplies amplitude $a_z$ by $e^{-i\gamma C(z)}$. The mixer applies a one-qubit rotation to every bit. `objective_function` computes $\sum_z |a_z|^2 C(z)$, and `optimize_parameters` minimizes this expectation with L-BFGS-B. `solve_vrp_quantum` then draws a finite sample and selects the shortest feasible sampled assignment. This last selection is a sampling and reporting step, not a proof that the assignment is globally optimal.

## 2. Create the illustrative instance

The coordinates and demands below define the example. Euclidean distance is measured in coordinate units; the notebook labels the resulting distance as kilometers only for continuity with the illustrative cost rate. The model does not include road networks, traffic, service times, or heterogeneous vehicles.

In [ ]:
depot_location = (50, 50)
customer_locations = [
    (20, 30), (70, 40), (60, 80), (30, 70),
    (80, 20), (40, 60), (75, 75), (25, 25),
]
customer_demands = [10, 15, 12, 8, 20, 18, 14, 16]
vehicle_capacity = 50
max_vehicles = 3

quantum_optimizer = QuantumInspiredQAOA(
    depot_location=depot_location,
    customer_locations=customer_locations,
    customer_demands=customer_demands,
    vehicle_capacity=vehicle_capacity,
    max_vehicles=max_vehicles,
)

print("Assignment-model instance")
print(f"Depot: {depot_location}")
print(f"Customers: {quantum_optimizer.n_customers}")
print(f"Total demand: {sum(customer_demands)}")
print(f"Vehicle capacity: {vehicle_capacity}")
print(f"Maximum vehicles: {max_vehicles}")
print(f"Qubits: {quantum_optimizer.n_qubits}")
print(f"Basis states: {len(quantum_optimizer.basis_states)}")
print(f"QAOA layers: {quantum_optimizer.n_layers}")
assert len(customer_locations) == len(customer_demands)
assert quantum_optimizer.costs.shape == (2**quantum_optimizer.n_qubits,)


## 3. Optimize the QAOA angles and sample assignments

The optimizer minimizes expected cost rather than directly selecting a single route. Measurement converts the final probability distribution into classical assignments. Since a finite sample may contain no feasible assignment, the result explicitly reports how many feasible samples were observed.

In [ ]:
quantum_solution = quantum_optimizer.solve_vrp_quantum(
    max_iterations=30,
    n_measurements=100,
)

print("QAOA-style assignment results")
print(f"Feasible samples: {quantum_solution['feasible_sample_count']} / 100")
print(f"Expected penalized cost: {quantum_solution['expected_cost']:.2f}")
print(f"Optimization time: {quantum_solution['optimization_time']:.2f} seconds")

if quantum_solution["best_state"] is None:
    print("No feasible assignment was observed in the measurement sample.")
else:
    for route_index, route in enumerate(quantum_solution["routes"], start=1):
        demand = sum(customer_demands[customer - 1] for customer in route)
        distance = quantum_optimizer.calculate_route_distance(route)
        print(f"Vehicle {route_index}: customers {route}, demand {demand}, distance {distance:.2f}")

    print(f"Total feasible sampled distance: {quantum_solution['total_distance']:.2f}")
    print(f"Illustrative cost at 2.5 per distance unit: {quantum_solution['total_cost']:.2f}")
    print(f"Selected state frequency: {quantum_solution['best_frequency']:.3f}")
    print(f"Selected bit string: {quantum_solution['best_state']}")
    for parameter_index, parameter in enumerate(quantum_solution["optimal_params"]):
        layer = parameter_index // 2 + 1
        parameter_name = "gamma (cost)" if parameter_index % 2 == 0 else "beta (mixer)"
        print(f"Layer {layer} {parameter_name}: {parameter:.4f} radians")

assert np.isclose(
    quantum_solution["final_state"].probabilities().sum(), 1.0, atol=1e-10
)
if quantum_solution["best_state"] is not None:
    decoded_routes, is_feasible = quantum_optimizer.decode_quantum_solution(
        quantum_solution["best_state"]
    )
    assert is_feasible
    assert sorted(customer for route in decoded_routes for customer in route) == list(
        range(1, len(customer_locations) + 1)
    )


## 4. Inspect the final probability distribution

The probability distribution describes measurement likelihoods, not a guarantee of solution quality. The entropy $H=-\sum_z p_z\log p_z$ summarizes how spread out the distribution is, and $e^H$ is its effective number of states. These statistics do not by themselves measure feasibility or optimality.

In [ ]:
final_state = quantum_solution["final_state"]
probabilities = final_state.probabilities()
positive_probabilities = probabilities[probabilities > 0]
entropy = float(-np.sum(positive_probabilities * np.log(positive_probabilities)))
effective_dimension = float(np.exp(entropy))

top_indices = np.argsort(probabilities)[-10:][::-1]
print("Most probable computational-basis states")
for rank, index in enumerate(top_indices, start=1):
    bit_string = final_state.basis_states[index]
    routes, feasible = quantum_optimizer.decode_quantum_solution(bit_string)
    print(
        f"{rank}. {bit_string} | probability={probabilities[index]:.5f} "
        f"| feasible={feasible} | nonempty routes={len(routes)}"
    )

print(f"Shannon entropy (nats): {entropy:.4f}")
print(f"Effective number of states: {effective_dimension:.2f}")
print(f"Total encoded states: {len(probabilities)}")
assert np.isclose(probabilities.sum(), 1.0, atol=1e-10)


## 5. Compare with a simple feasible construction

The greedy construction below places customers, in their original order, into the current vehicle until the next demand would exceed capacity, then opens another vehicle. It is included as a transparent reference construction, not as a competing optimization method or a proof of a lower bound. The QAOA result is compared only when at least one feasible assignment was measured.

In [ ]:
def greedy_solution(
    demands: List[int], capacity: int
) -> List[List[int]]:
    """Build feasible routes by sequentially packing customers in index order."""
    routes: List[List[int]] = []
    current_route: List[int] = []
    current_demand = 0

    for customer, demand in enumerate(demands, start=1):
        if demand > capacity:
            raise ValueError("A customer demand exceeds vehicle capacity.")
        if current_route and current_demand + demand > capacity:
            routes.append(current_route)
            current_route = []
            current_demand = 0
        current_route.append(customer)
        current_demand += demand

    if current_route:
        routes.append(current_route)
    return routes


greedy_routes = greedy_solution(customer_demands, vehicle_capacity)
greedy_distance = sum(
    quantum_optimizer.calculate_route_distance(route) for route in greedy_routes
)
greedy_cost = greedy_distance * 2.5
print(f"Greedy construction: {len(greedy_routes)} routes, distance {greedy_distance:.2f}")

if quantum_solution["best_state"] is not None:
    print(
        f"QAOA sampled assignment: {len(quantum_solution['routes'])} routes, "
        f"distance {quantum_solution['total_distance']:.2f}"
    )
    difference = greedy_distance - quantum_solution["total_distance"]
    if np.isclose(difference, 0.0, atol=1e-9):
        print("The two reported distances are equal within the comparison tolerance.")
    elif difference > 0:
        print(f"The sampled assignment is shorter by {difference:.2f} distance units.")
    else:
        print(f"The greedy construction is shorter by {-difference:.2f} distance units.")
else:
    print("A QAOA distance comparison is unavailable because no feasible sample was observed.")


## 6. Visualize routes and measurement probabilities

The route plot uses the customer order specified by the model. The probability plot displays the most probable basis states; labels show the leading portion of each bit string to keep the axis readable. A route result is plotted only when a feasible sampled assignment exists.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
route_ax, probability_ax = axes

route_ax.scatter(
    depot_location[0], depot_location[1], marker="s", color="black", s=100, label="Depot"
)
for customer_index, (x_coord, y_coord) in enumerate(customer_locations, start=1):
    route_ax.scatter(x_coord, y_coord, color="gray", s=45)
    route_ax.annotate(f"C{customer_index}", (x_coord, y_coord), xytext=(4, 4), textcoords="offset points")

if quantum_solution["best_state"] is not None:
    route_colors = plt.cm.Set2(np.linspace(0, 1, max(1, len(quantum_solution["routes"]))))
    for route_index, route in enumerate(quantum_solution["routes"]):
        coordinates = [depot_location] + [customer_locations[c - 1] for c in route] + [depot_location]
        route_ax.plot(
            [point[0] for point in coordinates],
            [point[1] for point in coordinates],
            marker="o",
            color=route_colors[route_index],
            label=f"Vehicle {route_index + 1}",
        )
route_ax.set_xlabel("X coordinate (distance units)")
route_ax.set_ylabel("Y coordinate (distance units)")
route_ax.set_title("Sampled feasible routes")
route_ax.grid(True, alpha=0.3)
route_ax.legend()

plot_count = min(20, len(probabilities))
top_indices = np.argsort(probabilities)[-plot_count:][::-1]
top_labels = [final_state.basis_states[index][:12] for index in top_indices]
top_probabilities = probabilities[top_indices]
probability_ax.barh(np.arange(plot_count), top_probabilities, color="steelblue")
probability_ax.set_yticks(np.arange(plot_count))
probability_ax.set_yticklabels(top_labels)
probability_ax.invert_yaxis()
probability_ax.set_xlabel("Measurement probability")
probability_ax.set_ylabel("Leading bits of basis-state label")
probability_ax.set_title("Most probable encoded states")
probability_ax.grid(True, axis="x", alpha=0.3)

fig.tight_layout()
plt.show()


## 7. Interpretation, limitations, and computational scaling

This experiment illustrates how a variational circuit can turn a classical cost function into a probability distribution over discrete assignments. The optimized expected cost is an average over that distribution; it is not the cost of the best sampled route. The reported feasible result is the shortest feasible assignment found in the finite measurement sample. A different random seed, optimizer trajectory, or sample can produce a different result, and multiple assignments may have the same distance.

Important modeling and computational limits are:

- **Fixed visit order:** route order is customer-index order, not an optimization decision. The result solves an assignment-and-fixed-order model rather than the general vehicle routing problem.
- **Penalty formulation:** invalid codes and capacity excess receive finite penalties. The decoder separately checks feasibility, so a low expected penalized cost must not be mistaken for a feasible solution.
- **No QUBO claim:** the exact capacity-excess penalty and fixed-order route cost are stored as a diagonal cost vector. They are not represented here as a quadratic unconstrained binary objective.
- **Classical simulation:** the state vector contains $2^{16}$ amplitudes for this instance. An exact state-vector simulator requires memory proportional to $2^n$ and applying one-qubit gates requires work proportional to $n2^n$. The notebook therefore demonstrates a small simulation, not a scalable quantum implementation.
- **No quantum-advantage evidence:** classical state-vector simulation and one illustrative comparison do not establish a speedup, quantum tunneling benefit, or performance advantage over classical optimization.

Extensions that preserve the same modeling workflow could change the cost definition to include additional operational requirements or study different small instances. Such extensions require a fresh analysis of the encoding, feasibility conditions, and simulator resource requirements.

In [ ]:
print("QAOA state-vector study complete")
print(f"Encoded binary variables: {quantum_optimizer.n_qubits}")
print(f"Basis states represented: {len(quantum_optimizer.basis_states)}")
print(f"Circuit layers: {quantum_optimizer.n_layers}")
print(f"Distinct measured states: {len(quantum_solution['measurements'])}")
print(f"Feasible measured samples: {quantum_solution['feasible_sample_count']} / 100")
if quantum_solution["best_state"] is not None:
    print(f"Best feasible sampled distance: {quantum_solution['total_distance']:.2f}")
else:
    print("No feasible measured assignment was available for reporting.")
